# Lumis 20: make v3 fast enough for the app (8-bit model)

Notebook 19 exported v3, but the MedSigLIP image encoder takes **12.5 seconds per photo** on Colab's 2-core CPU,
which is too slow for the app. The usual fix is **8-bit quantization**: the encoder's weights are stored as 8-bit
integers instead of 32-bit numbers, so the file is about 4 times smaller and the CPU does less work. The classifier
on top (scaler, PCA, logistic regression) is tiny and stays as it is. Nothing is retrained.

The risk is that rounding the weights changes some predictions, so the 8-bit model is only adopted if it behaves
like the full model.

**What it does**
1. Makes the 8-bit encoder (ONNX Runtime dynamic quantization of the matrix multiplications, per-channel weights).
2. Times the full and the 8-bit encoder on the same 16 photos on the CPU.
3. Compares their results on **development photos** (decision set): about 290 photos, up to 32 per condition from the
   SCIN train + validation cases and the DermNet validation split. Both use the app's preprocessing (PIL resize).
4. Only if the rules pass: a guard check on the 121 SCIN test cases, then saves the 8-bit encoder and updates the v3
   config so the backend uses it.

**Rules fixed before running (2026-11-14).** The 8-bit encoder is adopted only if, on the decision set, compared with
the full encoder:
1. the top condition is the same on **at least 98%** of photos;
2. the confidence band (high / moderate / refer) is the same on **at least 95%**;
3. the top probability moves by **at most 0.05** on 95% of photos;
4. it is **at least 1.5 times faster** on the CPU (median time per photo);
and on the SCIN test cases (guard, checked only for the variant that passed 1 to 4) accuracy and average sensitivity
are each **within 2 points** of the full encoder.

Variants are tried in a fixed order and the first one that passes is used: **A** signed 8-bit weights (ONNX Runtime's
default), then **B** unsigned 8-bit weights (avoids an overflow some CPUs have with A). If neither passes, the app
keeps the full encoder and the speed has to come from the server (more CPU cores).

**Runtime: GPU (T4)**, used only to run the full encoder quickly as the reference; the 8-bit encoder runs on the CPU,
as it will on the server. No secrets needed. Start from a fresh runtime. About 30 to 45 minutes; the 8-bit results are
saved to Drive as they go, so a rerun continues where it stopped.

In [ ]:
SEED = 42
PER_CLASS = 32                       # decision set: up to this many photos per condition from each source
N_SPEED = 16                         # photos used for timing
MIN_SAME_PRED, MIN_SAME_BAND, MAX_P95_PROB_SHIFT, MIN_SPEEDUP = 0.98, 0.95, 0.05, 1.5
GUARD_POINTS = 0.02
VARIANTS = ['A_s8', 'B_u8']           # tried in this order; the first one that passes is used

In [ ]:
!pip install -q onnx onnxruntime-gpu
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/Lumis'

In [ ]:
import os, json, time, shutil, zipfile, hashlib
import numpy as np, pandas as pd
import torch                                   # loads the CUDA libraries ONNX Runtime needs
import onnxruntime as ort
from PIL import Image, ImageOps

M = f'{BASE}/exports/model_v3'
backup = f'{M}/lumis_model_config_v3_fp32.json'               # exists only if a previous run adopted the 8-bit encoder
cfg = json.load(open(backup if os.path.exists(backup) else f'{M}/lumis_model_config_v3.json'))
assert cfg['version'] == 'v3', 'Run notebook 19 first.'
CLASSES, K = cfg['classes'], len(cfg['classes'])
IDX = {c: i for i, c in enumerate(CLASSES)}
SCABIES = IDX['scabies']
ADJ, T, BANDS, PRE = np.array(cfg['logit_adjust']), cfg['temperature'], cfg['bands'], cfg['preprocessing']
h = np.load(f'{M}/{cfg["head_file"]}')

FP32 = '/content/lumis_medsiglip_v3.onnx'
if not os.path.exists(FP32):
    shutil.copy(f'{M}/{cfg["onnx_file"]}', FP32)
assert hashlib.sha256(open(FP32, 'rb').read()).hexdigest() == cfg['onnx_sha256'], 'ONNX file does not match the v3 config.'
print('ONNX Runtime', ort.__version__, '| providers', ort.get_available_providers(), '| CPU cores', os.cpu_count())

## 1. Photos (decision set, speed photos, SCIN test cases)

In [ ]:
derm = pd.read_csv(f'{BASE}/data/lumis_split_v2.csv')
d_val = derm[derm.split == 'val'].reset_index(drop=True)
split = pd.read_csv(f'{BASE}/data/scin_split_v1.csv', dtype={'case_id': str})
ev = split[split.split.isin(['train', 'val']) & split.eval_photo].reset_index(drop=True)      # one photo per case
scin_test = split[(split.split == 'test') & split.eval_photo].reset_index(drop=True)

def sample(df, source):
    parts = [g.sample(min(len(g), PER_CLASS), random_state=SEED) for _, g in df.groupby('lumis_class')]
    return pd.concat(parts).assign(source=source)[['path', 'lumis_class', 'source']]

dec = pd.concat([sample(ev, 'scin'), sample(d_val, 'dermnet')]).reset_index(drop=True)
scin_test = scin_test.assign(source='scin')[['path', 'lumis_class', 'source']]

# extract only the photos needed (not the whole zips)
def extract(df):
    files = []
    for src, z, dest in [('dermnet', f'{BASE}/data/dermnet.zip', '/content/dermnet'), ('scin', f'{BASE}/data/scin_images.zip', '/content')]:
        sub = df[df.source == src]
        with zipfile.ZipFile(z) as f:
            for p in sub.path:
                if not os.path.exists(f'{dest}/{p}'):
                    f.extract(p, dest)
    return [(f'/content/dermnet/{p}' if s == 'dermnet' else f'/content/{p}') for p, s in zip(df.path, df.source)]

dec['file'], scin_test['file'] = extract(dec), extract(scin_test)
def readable(f):
    try:
        Image.open(f).verify()
        return True
    except Exception:
        print('cannot be read, left out:', f)
        return False
dec = dec[dec.file.map(readable)].reset_index(drop=True)
scin_test = scin_test[scin_test.file.map(readable)].reset_index(drop=True)
speed_files = list(dec.sample(N_SPEED, random_state=SEED).file)
print(f'Decision set {len(dec)} photos:')
print(pd.crosstab(dec.lumis_class, dec.source, margins=True))
print(f'SCIN test {len(scin_test)} cases | speed photos {len(speed_files)}')

## 2. App preprocessing and classifier (exactly what the backend will do)

In [ ]:
def open_rgb(path):
    return ImageOps.exif_transpose(Image.open(path)).convert('RGB')

def preprocess(img):
    hgt, wid = PRE['size']
    a = np.asarray(img.resize((wid, hgt), resample=PRE['resample']), dtype=np.float32) * PRE['rescale_factor']
    a = (a - np.array(PRE['image_mean'], np.float32)) / np.array(PRE['image_std'], np.float32)
    return a.transpose(2, 0, 1)[None]

def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def probs(F):
    Z = ((F - h['scaler_mean']) / h['scaler_scale'] - h['pca_mean']) @ h['pca_components'].T
    return softmax((Z @ h['coef'].T + h['intercept'] + ADJ) / T)

def bands(p):
    c, pred = p.max(1), p.argmax(1)
    b = np.where(c >= BANDS['high_from'], 'high', np.where(c >= BANDS['refer_below'], 'moderate', 'refer'))
    return np.where((pred == SCABIES) & (b == 'high'), 'moderate', b)

def macro_sens(y, pred):
    return float(np.mean([(pred[y == k] == k).mean() for k in range(K) if (y == k).any()]))

def cpu_session(path):
    so = ort.SessionOptions()
    so.intra_op_num_threads = os.cpu_count()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(path, so, providers=['CPUExecutionProvider'])

def run(sess, files, cache=None):
    """Embeddings for files, saved in chunks of 32 to Drive when cache is given (so a rerun continues)."""
    out = []
    for i in range(0, len(files), 32):
        chunk_file = f'{cache}_{i // 32:04d}.npy' if cache else None
        if chunk_file and os.path.exists(chunk_file):
            out.append(np.load(chunk_file))
            continue
        e = np.concatenate([sess.run(None, {'pixel_values': preprocess(open_rgb(f))})[0] for f in files[i:i + 32]])
        if chunk_file:
            np.save(chunk_file, e)
        out.append(e)
        print(f'  {min(i + 32, len(files))}/{len(files)}', end='\r')
    return np.concatenate(out).astype(np.float32)

def seconds_per_photo(sess):
    x = [preprocess(open_rgb(f)) for f in speed_files]
    for xi in x[:2]:
        sess.run(None, {'pixel_values': xi})                     # warm-up
    t = []
    for xi in x:
        t0 = time.perf_counter()
        sess.run(None, {'pixel_values': xi})
        t.append(time.perf_counter() - t0)
    return float(np.median(t))

## 3. Reference: the full encoder (GPU for the comparisons, CPU for the timing)

In [ ]:
assert 'CUDAExecutionProvider' in ort.get_available_providers(), 'No GPU for ONNX Runtime: switch the runtime to T4 GPU and run all again.'
gpu = ort.InferenceSession(FP32, providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
assert gpu.get_providers()[0] == 'CUDAExecutionProvider', 'ONNX Runtime fell back to the CPU: send this to Claude.'
t0 = time.time()
F_dec, F_test = run(gpu, list(dec.file)), run(gpu, list(scin_test.file))
del gpu
print(f'Full encoder embeddings for {len(dec) + len(scin_test)} photos in {time.time() - t0:.0f} s')

# the reference must reproduce notebook 19's SCIN test result (same photos; there the encoder was PyTorch on the GPU)
y_test = scin_test.lumis_class.map(IDX).values
p_ref_test = probs(F_test)
saved = json.load(open(f'{BASE}/exports/final_test_v3/metrics.json'))['scin_test']['v3']
print(f'SCIN test, full encoder here: accuracy {(p_ref_test.argmax(1) == y_test).mean():.3f} '
      f'(notebook 19: {saved["accuracy"]:.3f}), average sensitivity {macro_sens(y_test, p_ref_test.argmax(1)):.3f} '
      f'(notebook 19: {saved["macro_sensitivity"]:.3f})')

t_fp32 = seconds_per_photo(cpu_session(FP32))
print(f'Full encoder on the CPU: {t_fp32:.2f} s per photo ({os.cpu_count()} cores)')

## 4. 8-bit variants, in the fixed order

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

OUT = f'{BASE}/exports/speed'
os.makedirs(OUT, exist_ok=True)
y_dec = dec.lumis_class.map(IDX).values
p_ref, b_ref = probs(F_dec), bands(probs(F_dec))
WEIGHT = {'A_s8': QuantType.QInt8, 'B_u8': QuantType.QUInt8}
KEY = hashlib.md5(('\n'.join(dec.file) + '|' + '\n'.join(scin_test.file) + '|' + cfg['onnx_sha256']).encode()).hexdigest()[:10]

results, chosen = {}, None
for v in VARIANTS:
    path = f'/content/lumis_medsiglip_v3_{v}.onnx'
    if not os.path.exists(path):
        t0 = time.time()
        quantize_dynamic(FP32, path, weight_type=WEIGHT[v], per_channel=True, op_types_to_quantize=['MatMul'])
        print(f'{v}: quantized in {time.time() - t0:.0f} s')
    sess = cpu_session(path)
    t_v = seconds_per_photo(sess)
    F_v = run(sess, list(dec.file), cache=f'{OUT}/{v}_{KEY}_dec')
    p_v = probs(F_v)
    r = {'size_gb': os.path.getsize(path) / 1e9, 'seconds_per_photo': t_v, 'speedup': t_fp32 / t_v,
         'same_prediction': float((p_v.argmax(1) == p_ref.argmax(1)).mean()),
         'same_band': float((bands(p_v) == b_ref).mean()),
         'p95_top_prob_shift': float(np.quantile(np.abs(p_v.max(1) - p_ref.max(1)), 0.95)),
         'accuracy_full': float((p_ref.argmax(1) == y_dec).mean()), 'accuracy_8bit': float((p_v.argmax(1) == y_dec).mean())}
    r['checks'] = {'same_prediction': r['same_prediction'] >= MIN_SAME_PRED, 'same_band': r['same_band'] >= MIN_SAME_BAND,
                   'prob_shift': r['p95_top_prob_shift'] <= MAX_P95_PROB_SHIFT, 'speedup': r['speedup'] >= MIN_SPEEDUP}
    r['passed'] = all(r['checks'].values())
    results[v] = r
    print(f"\n{v}: {r['size_gb']:.2f} GB, {t_v:.2f} s per photo ({r['speedup']:.1f}x faster) | same prediction "
          f"{r['same_prediction']:.1%} | same band {r['same_band']:.1%} | top probability shift (95th pct) "
          f"{r['p95_top_prob_shift']:.3f} | accuracy on the decision set {r['accuracy_full']:.3f} -> {r['accuracy_8bit']:.3f}")
    print('  checks:', r['checks'], '->', 'PASSED' if r['passed'] else 'not passed')
    if r['passed']:
        chosen, chosen_path, chosen_sess = v, path, sess
        break
print('\nChosen:', chosen or 'none (the app keeps the full encoder)')

## 5. Guard check on the SCIN test cases (only for the chosen variant)

In [ ]:
guard = None
if chosen:
    p_v = probs(run(chosen_sess, list(scin_test.file), cache=f'{OUT}/{chosen}_{KEY}_test'))
    pr, pv = p_ref_test.argmax(1), p_v.argmax(1)
    guard = {'accuracy_full': float((pr == y_test).mean()), 'accuracy_8bit': float((pv == y_test).mean()),
             'macro_sens_full': macro_sens(y_test, pr), 'macro_sens_8bit': macro_sens(y_test, pv),
             'same_prediction': float((pr == pv).mean()), 'n': int(len(y_test)),
             **{f'sens_{c}_8bit': float((pv[y_test == k] == k).mean()) for k, c in enumerate(CLASSES) if (y_test == k).any()}}
    guard['passed'] = bool(guard['accuracy_8bit'] >= guard['accuracy_full'] - GUARD_POINTS - 1e-9
                           and guard['macro_sens_8bit'] >= guard['macro_sens_full'] - GUARD_POINTS - 1e-9)
    print(json.dumps({k: round(v, 3) if isinstance(v, float) else v for k, v in guard.items()}, indent=1))
    print('Guard', 'PASSED' if guard['passed'] else 'NOT passed: the app keeps the full encoder')
adopted = bool(chosen and guard['passed'])

## 6. Save

In [ ]:
decision = {'adopted': adopted, 'variant': chosen if adopted else None, 'full_seconds_per_photo': t_fp32,
            'cpu_cores': os.cpu_count(), 'variants': results, 'guard_scin_test': guard,
            'rules': {'min_same_prediction': MIN_SAME_PRED, 'min_same_band': MIN_SAME_BAND,
                      'max_p95_top_prob_shift': MAX_P95_PROB_SHIFT, 'min_speedup': MIN_SPEEDUP, 'guard_points': GUARD_POINTS},
            'decision_set': {'n': int(len(dec)), 'by_source': dec.source.value_counts().to_dict()},
            'onnxruntime': ort.__version__, 'created_at': time.strftime('%Y-%m-%d %H:%M')}
json.dump(decision, open(f'{OUT}/decision.json', 'w'), indent=1, default=float)

new_cfg = {**cfg, 'preprocessing': {**cfg['preprocessing'], 'resize_library': 'PIL'}}
if adopted:
    name = 'lumis_medsiglip_v3_int8.onnx'
    shutil.copy(chosen_path, f'{M}/{name}')
    if not os.path.exists(backup):
        shutil.copy(f'{M}/lumis_model_config_v3.json', backup)
    new_cfg.update({'onnx_file': name, 'onnx_sha256': hashlib.sha256(open(chosen_path, 'rb').read()).hexdigest(),
                    'encoder': 'google/medsiglip-448 image encoder (ONNX, 8-bit dynamic quantization)',
                    'onnx_full_file': cfg['onnx_file'], 'onnx_full_sha256': cfg['onnx_sha256'],
                    'quantization': {'variant': chosen, 'method': 'onnxruntime quantize_dynamic, MatMul, per-channel',
                                     'seconds_per_photo': results[chosen]['seconds_per_photo'], 'full_seconds_per_photo': t_fp32,
                                     'same_prediction_dev': results[chosen]['same_prediction'], 'scin_test_guard': guard}})
json.dump(new_cfg, open(f'{M}/lumis_model_config_v3.json', 'w'), indent=1, default=float)
print('Adopted:', adopted, '| encoder in the config:', new_cfg['onnx_file'])
print('Saved', f'{OUT}/decision.json', 'and', f'{M}/lumis_model_config_v3.json')
print(os.listdir(M))